# DeepSets Hyperparameter Tuning

This notebook implements hyperparameter search for the DeepSets quantum error correction decoder.

**Architecture:** Using coordinate-based features (x, y, t) for fired detectors.

**Workflow:**
1. Set `WORKER_ID` below (1-5), or use `CUSTOM_CONFIGS` to specify exact config IDs
2. Run the training loop to train the assigned configurations
3. After all workers complete, run the analysis cells

**Data:** Uses d=7 dataset from `code/nn_datasets/d7_baseline_array.pt`

**Epochs:** 50 epochs per configuration (matching gSAGE protocol)

In [ ]:
#==============================================================================
# CONFIGURATION
#==============================================================================
WORKER_ID = 1  # Set to 1, 2, 3, 4, or 5 for each instance

# Optional: specify custom config IDs to train (in order). If set, overrides WORKER_ID.
CUSTOM_CONFIGS = []  # e.g., [5, 12, 23, 7]
#==============================================================================

# Determine which configs to train
if CUSTOM_CONFIGS:
    my_config_ids = CUSTOM_CONFIGS
    print(f"Using CUSTOM_CONFIGS: {my_config_ids}")
else:
    assert WORKER_ID in [1, 2, 3, 4, 5], f"WORKER_ID must be 1-5, got {WORKER_ID}"
    my_config_ids = [i for i in range(50) if i % 5 == (WORKER_ID - 1)]
    print(f"Worker ID: {WORKER_ID}")

print(f"This worker will train {len(my_config_ids)} configs: {my_config_ids}")

## Imports and Setup

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime
from itertools import product

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/tuning -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Import DeepSets from benchmark_models
from benchmark_models import DeepSets, SurfaceCodeSampler

# Setup paths
TUNING_DIR = BASE_PATH / "deepsets" / "tuning"
RESULTS_DIR = TUNING_DIR / "results"
MODELS_DIR = TUNING_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TUNING_DIR: {TUNING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

## Hyperparameter Search Space

In [ ]:
# Define hyperparameter search space
SEARCH_SPACE = {
    'phi_hidden': [
        (64, 64),
        (128, 128),
        (256, 256),
        (128, 256),
        (64, 128, 64),
    ],
    'rho_hidden': [
        (64,),
        (128,),
        (128, 64),
        (256, 128),
        (256, 128, 64),
    ],
    'pool': ['sum', 'mean'],
    'lr': [1e-3, 5e-4],
}

# Generate all configurations
all_configs = []
for phi, rho, pool, lr in product(
    SEARCH_SPACE['phi_hidden'],
    SEARCH_SPACE['rho_hidden'],
    SEARCH_SPACE['pool'],
    SEARCH_SPACE['lr']
):
    all_configs.append({
        'phi_hidden': phi,
        'rho_hidden': rho,
        'pool': pool,
        'lr': lr,
    })

# Truncate to 50 configs for worker distribution
all_configs = all_configs[:50]

print(f"Total configurations: {len(all_configs)}")
print(f"\nExample config 0: {all_configs[0]}")
print(f"Example config 1: {all_configs[1]}")

## Load Dataset

In [ ]:
# Training configuration
D = 7  # Code distance for tuning
EPOCHS = 50
BATCH_SIZE = 256
SEED = 42

# Dataset split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Load dataset
DATASET_PATH = BASE_PATH / "nn_datasets" / f"d{D}_baseline_array.pt"
print(f"Loading dataset from: {DATASET_PATH}")

data = torch.load(DATASET_PATH, weights_only=False)
detections = data['detections'].float()
labels = data['labels'].float()

print(f"Dataset size: {len(labels):,} samples")
print(f"Detections shape: {detections.shape}")

# Shuffle and split
random.seed(SEED)
torch.manual_seed(SEED)

n = len(labels)
perm = torch.randperm(n)

train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_det = detections[perm[:train_end]]
train_labels = labels[perm[:train_end]]
val_det = detections[perm[train_end:val_end]]
val_labels = labels[perm[train_end:val_end]]
test_det = detections[perm[val_end:]]
test_labels = labels[perm[val_end:]]

print(f"\nSplit:")
print(f"  Train: {len(train_labels):,}")
print(f"  Val: {len(val_labels):,}")
print(f"  Test: {len(test_labels):,}")

## Training Functions

In [ ]:
def evaluate_accuracy(model, detections, labels, d, batch_size=512):
    """Evaluate accuracy on a dataset."""
    model.model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch_det = detections[i:i+batch_size].to(device)
            batch_labels = labels[i:i+batch_size].to(device)
            
            preds = model.predict(batch_det, d)
            correct += ((preds > 0.5).float() == batch_labels).sum().item()
            total += len(batch_labels)
    
    return correct / total


def train_config(config_id, config, verbose=True):
    """Train a single configuration."""
    result_path = RESULTS_DIR / f"config_{config_id:03d}.json"
    
    # Check if already completed
    if result_path.exists():
        if verbose:
            print(f"Config {config_id} already completed, skipping.")
        return None
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Training Config {config_id}")
        print(f"{'='*60}")
        print(f"  phi_hidden: {config['phi_hidden']}")
        print(f"  rho_hidden: {config['rho_hidden']}")
        print(f"  pool: {config['pool']}")
        print(f"  lr: {config['lr']}")
    
    start_time = time.time()
    
    # Initialize model
    model = DeepSets(
        nickname=f"config_{config_id}",
        phi_hidden=config['phi_hidden'],
        rho_hidden=config['rho_hidden'],
        pool=config['pool'],
        dropout=0.0,
        device=device,
        base_path=BASE_PATH,
        seed=SEED
    )
    
    # Train
    losses = model.train_from_data(
        detections=train_det,
        labels=train_labels,
        d=D,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=config['lr'],
        verbose=verbose
    )
    
    train_time = time.time() - start_time
    
    # Evaluate
    val_acc = evaluate_accuracy(model, val_det, val_labels, D)
    test_acc = evaluate_accuracy(model, test_det, test_labels, D)
    
    if verbose:
        print(f"\nResults:")
        print(f"  Val accuracy: {val_acc*100:.2f}%")
        print(f"  Test accuracy: {test_acc*100:.2f}%")
        print(f"  Training time: {train_time/60:.1f} min")
    
    # Save results
    result = {
        'config_id': config_id,
        'config': config,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'final_loss': losses[-1],
        'training_time_seconds': train_time,
        'epochs': EPOCHS,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    
    # Save model
    model_path = MODELS_DIR / f"config_{config_id:03d}.pt"
    torch.save({
        'state_dict': model.model.state_dict(),
        'config': model._config,
        'hyperparams': config,
    }, model_path)
    
    return result

## Run Training Loop

In [ ]:
# Train assigned configurations
results = []

for config_id in my_config_ids:
    if config_id >= len(all_configs):
        print(f"Config {config_id} out of range, skipping.")
        continue
    
    config = all_configs[config_id]
    result = train_config(config_id, config)
    if result:
        results.append(result)

print(f"\n{'='*60}")
print(f"Training complete! Trained {len(results)} configs.")
print(f"{'='*60}")

## Analysis (Run after all workers complete)

In [ ]:
# Collect all results
all_results = []
for result_file in sorted(RESULTS_DIR.glob("config_*.json")):
    with open(result_file, 'r') as f:
        all_results.append(json.load(f))

print(f"Collected {len(all_results)} results")

if all_results:
    # Sort by validation accuracy
    all_results.sort(key=lambda x: x['val_accuracy'], reverse=True)
    
    print("\nTop 5 Configurations:")
    print("-" * 80)
    for i, r in enumerate(all_results[:5]):
        print(f"{i+1}. Config {r['config_id']}: Val={r['val_accuracy']*100:.2f}%, Test={r['test_accuracy']*100:.2f}%")
        print(f"   phi={r['config']['phi_hidden']}, rho={r['config']['rho_hidden']}, pool={r['config']['pool']}, lr={r['config']['lr']}")
    
    # Save best config
    best = all_results[0]
    best_config_path = TUNING_DIR / "best_model_config.json"
    with open(best_config_path, 'w') as f:
        json.dump({
            'config_id': best['config_id'],
            'model_params': {
                'phi_hidden': best['config']['phi_hidden'],
                'rho_hidden': best['config']['rho_hidden'],
                'pool': best['config']['pool'],
                'dropout': 0.0,
            },
            'training_params': {
                'learning_rate': best['config']['lr'],
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
            },
            'performance': {
                'val_accuracy': best['val_accuracy'],
                'test_accuracy': best['test_accuracy'],
            },
            'timestamp': datetime.now().isoformat()
        }, f, indent=2)
    print(f"\nBest config saved to: {best_config_path}")